<a href="https://colab.research.google.com/github/Katona-lab/teaching/blob/main/Correlated_Diffraction_Bayesian_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Modeling Multivariate Distributions in X‑ray Diffraction Data: Colab Practical

This self‑contained notebook introduces Bayesian modeling of correlated diffraction intensities. You will simulate data, fit univariate and multivariate models in PyMC, and compare posterior uncertainties. The content follows the teaching outline from prior practicals in our group and two papers listed below.

**Key outcomes**
- Construct and sample simple Bayesian models in PyMC
- Interpret posterior distributions for correlated measurements
- Understand when multivariate treatment improves inference

**Environment**
- Python ≥ 3.10
- Packages: `numpy`, `matplotlib`, `pymc`, `arviz`

**Note**: All data are generated in code. No external files are required.



## References

- Katona, G., Garcia‑Bonete, M. J., & Lundholm, I. V. (2016). *Acta Cryst. A72*, 406–411. Estimating the difference between structure‑factor amplitudes using multivariate Bayesian inference.
- Garcia‑Bonete, M. J., & Katona, G. (2019). *Acta Cryst. A75*, 851–860. Bayesian machine learning improves single‑wavelength anomalous diffraction phasing.



## 0. Setup

Run the cell below in Colab to install the required libraries.


In [ ]:

# If running on Colab, uncomment the next line
!pip -q install numpy matplotlib pymc arviz


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

import pymc as pm
import arviz as az

rng = np.random.default_rng(42)
az.rcParams["stats.hdi_prob"] = 0.95

print("Library versions:")
print("numpy:", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)
print("pymc:", pm.__version__)
print("arviz:", az.__version__)



## 1. Simulating correlated intensities

We simulate two intensity measurements \($I_1, I_2$\) of related reflections. Each observation is a sum of a true mean \($F^2$\) plus a shared systematic term and an independent random term:


\begin{aligned}
I_1 &= F_1^2 + \varepsilon_{\text{sys}} + \varepsilon_{\text{ran,1}}\\
I_2 &= F_2^2 + \varepsilon_{\text{sys}} + \varepsilon_{\text{ran,2}}\\
\varepsilon_{\text{sys}} &\sim \mathcal N(0,\sigma_{\text{sys}})\\
\varepsilon_{\text{ran},j} &\sim \mathcal N(0,\sigma_{\text{ran}})\;.
\end{aligned}


This construction induces positive correlation between \($I_1$\) and \($I_2$\).

In [ ]:

def simulate_pairs(n=20, F1_true=0.1, F2_true=0.7, sigma_sys=1.0, sigma_ran=0.3, seed=42):
    rng = np.random.default_rng(seed)
    eps_sys = rng.normal(0, sigma_sys, size=n)
    eps_r1  = rng.normal(0, sigma_ran, size=n)
    eps_r2  = rng.normal(0, sigma_ran, size=n)
    I1 = F1_true**2 + eps_sys + eps_r1
    I2 = F2_true**2 + eps_sys + eps_r2
    return I1, I2

I1, I2 = simulate_pairs(n=20, F1_true=0.1, F2_true=0.7, sigma_sys=1.0, sigma_ran=0.3, seed=2025)

r = np.corrcoef(I1, I2)[0,1]
print(f"Sample correlation r = {r:.3f}")


In [ ]:

# Scatter plot of intensities
plt.figure()
plt.scatter(I1, I2, s=60)
plt.xlabel("I1")
plt.ylabel("I2")
plt.title("Simulated correlated intensities")
plt.grid(True)
plt.show()



**Question**: What happens to the scatter pattern when \($\sigma_{\text{sys}}$\) increases?

**Answer**: The points align more strongly along a diagonal ridge since a larger shared component increases correlation. The cloud becomes more elongated with smaller spread perpendicular to the diagonal, while marginal spreads increase together.



### 1.1 Varying the correlation driver

The function below sweeps \($\sigma_{\text{sys}}$\) to illustrate how the induced correlation changes.


In [ ]:

sigmas = np.linspace(0.0, 1.5, 10)
corrs  = []
for ssys in sigmas:
    I1_tmp, I2_tmp = simulate_pairs(n=2000, sigma_sys=ssys, sigma_ran=0.3, seed=123)
    corrs.append(np.corrcoef(I1_tmp, I2_tmp)[0,1])

plt.figure()
plt.plot(sigmas, corrs, marker="o")
plt.xlabel("sigma_sys")
plt.ylabel("sample correlation r")
plt.title("Induced correlation as sigma_sys increases")
plt.grid(True)
plt.show()



## 2. Univariate Bayesian estimation

We first ignore correlation and treat \($I_1$\) and \($I_2$\) as independent normals with shared noise scale \($\sigma$\):


$I_j \sim \mathcal N(F_j^2,\,\sigma),\quad j \in \{1,2\}.
$

Priors:
\($F_1, F_2 \sim \text{Uniform}(0,10)$\), \($\sigma \sim \text{HalfNormal}(1)$\).


In [ ]:

with pm.Model() as uni_model:
    F1 = pm.Uniform('F1', 0, 10)
    F2 = pm.Uniform('F2', 0, 10)
    sigma1 = pm.HalfNormal('sigma1', 1.0)
    sigma2 = pm.HalfNormal('sigma2', 1.0)
    I1_obs = pm.Normal('I1_obs', mu=F1**2, sigma=sigma1, observed=I1)
    I2_obs = pm.Normal('I2_obs', mu=F2**2, sigma=sigma2, observed=I2)
    trace_uni = pm.sample(2000, tune=1000, chains=2, target_accept=0.9, progressbar=True)


In [ ]:

az.plot_trace(trace_uni, var_names=["F1","F2","sigma1","sigma2"])
plt.show()

az.plot_posterior(trace_uni, var_names=["F1","F2","sigma1","sigma2"], point_estimate="mean")
plt.show()

print(az.summary(trace_uni, var_names=["F1","F2","sigma1","sigma2"]).round(3))



### 3.1 Posterior visualization of univariate model

The scatter plot below displays samples of $F_1$ and $F_2$ to show their posterior.

In [ ]:
samples = az.extract(trace_uni, var_names=["F1","F2"]).to_dataframe().values
plt.figure()
plt.plot([0,1.5],[0,1.5])
plt.scatter(samples[:,0], samples[:,1], s=5, alpha=0.2)
plt.xlabel("F1")
plt.ylabel("F2")
plt.plot([0,0],[1.5,1.5])
plt.title("Joint posterior samples for F1 and F2 (multivariate model)")
plt.grid(True)
plt.show()


## 3. Multivariate Bayesian estimation

Now we account for correlation by modeling the pair
$ I_1, I_2 $
with a bivariate normal:

$
\begin{bmatrix} I_1 \\ I_2 \end{bmatrix} \sim \mathcal N\left(
\begin{bmatrix} F_1^2 \\ F_2^2 \end{bmatrix},
\begin{bmatrix}
\sigma^2 & \rho\,\sigma^2 \\
\rho\,\sigma^2 & \sigma^2
\end{bmatrix}\right).
$

Priors:
\($F_1, F_2 \sim \text{Uniform}(0,10)$\), \($\sigma \sim \text{HalfNormal}(1)$\), \($\rho \sim \text{Uniform}(-1,1)$\).


In [ ]:

Y = np.column_stack([I1, I2])

with pm.Model() as mv_model:
    F1 = pm.Uniform('F1', 0, 10)
    F2 = pm.Uniform('F2', 0, 10)
    sigma = pm.HalfNormal('sigma', 1.0, shape=2)
    rho = pm.Uniform('rho', -1, 1)
    cov = pm.math.stack([[sigma[0]**2, rho*sigma[0]*sigma[1]],
                         [rho*sigma[0]*sigma[1], sigma[1]**2]])
    mu = pm.math.stack([F1**2, F2**2])
    I_obs = pm.MvNormal('I_obs', mu=mu, cov=cov, observed=Y)
    trace_mv = pm.sample(2000, tune=1000, chains=2, target_accept=0.9, progressbar=True)


In [ ]:

az.plot_trace(trace_mv, var_names=["F1","F2","sigma","rho"])
plt.show()

az.plot_posterior(trace_mv, var_names=["F1","F2","sigma","rho"], point_estimate="mean")
plt.show()

print(az.summary(trace_mv, var_names=["F1","F2","sigma","rho"]).round(3))



### 3.1 Joint posterior visualization

The scatter plot below displays joint samples of \($(F_1, F_2)$\) to show their posterior dependence under the multivariate model. The diagonal corresponds to unchanged structure factor amplitudes \($F_1=F_2$\).


In [ ]:

samples = az.extract(trace_mv, var_names=["F1","F2"]).to_dataframe().values

plt.figure()
plt.plot([0,1.5],[0,1.5])
plt.scatter(samples[:,0], samples[:,1], s=5, alpha=0.2)
plt.xlabel("F1")
plt.ylabel("F2")
plt.plot([0,0],[1.5,1.5])
plt.title("Joint posterior samples for F1 and F2 (multivariate model)")
plt.grid(True)
plt.show()



## 4. Comparing uncertainty for the amplitude difference

We compare the posterior uncertainty of the difference $(Δ F = F_2 - F_1)$ under the univariate and multivariate models.


In [ ]:

def delta_hdi_width(trace, var1="F1", var2="F2"):
    s = az.extract(trace, var_names=[var1, var2]).to_dataframe()
    df = np.array(s[var2] - s[var1])
    hdi = az.hdi(df, hdi_prob=0.95)
    return float(hdi[1] - hdi[0])

w_uni = delta_hdi_width(trace_uni)
w_mv  = delta_hdi_width(trace_mv)

print(f"95% HDI width for ΔF (univariate):   {w_uni:.4f}")
print(f"95% HDI width for ΔF (multivariate): {w_mv:.4f}")



## 5. Effect of correlation strength in simulations

We sweep a target correlation by controlling the ratio of systematic and random variances, fit both models for each setting, and track the 95 percent HDI width of \($\Delta F$\).

This block may take several minutes in Colab. Reduce `n_pairs` or `n_draws` if needed.


In [ ]:

def simulate_with_rho(n_pairs=30, F1_true=0.1, F2_true=0.7, sigma_ran=0.3, rho_target=0.0, seed=123):
    # Given sigma_ran and target rho, choose sigma_sys so that Corr(I1,I2) ≈ rho_target
    # For the additive model, Var(Ij) = sigma_sys^2 + sigma_ran^2 and Cov = sigma_sys^2
    # So rho = sigma_sys^2 / (sigma_sys^2 + sigma_ran^2)
    if rho_target <= 0:
        sigma_sys = 0.0
    else:
        sigma_sys = np.sqrt((rho_target * sigma_ran**2) / (1 - rho_target))
        print(sigma_sys,sigma_ran)
    return simulate_pairs(n_pairs, F1_true, F2_true, sigma_sys, sigma_ran, seed)

def fit_uni(I1, I2, draws=1000, tune=500):
    with pm.Model() as m:
        F1 = pm.Uniform('F1', 0, 10)
        F2 = pm.Uniform('F2', 0, 10)
        sigma = pm.HalfNormal('sigma', 1.0)
        pm.Normal('I1_obs', mu=F1**2, sigma=sigma, observed=I1)
        pm.Normal('I2_obs', mu=F2**2, sigma=sigma, observed=I2)
        tr = pm.sample(draws, tune=tune, chains=2, target_accept=0.9, progressbar=False)
    return tr

def fit_mv(I1, I2, draws=1000, tune=500):
    Y = np.column_stack([I1, I2])
    with pm.Model() as m:
        F1 = pm.Uniform('F1', 0, 10)
        F2 = pm.Uniform('F2', 0, 10)
        #sigma = pm.HalfNormal('sigma', 1.0, shape=2)
        sigma = pm.HalfNormal('sigma', 1.0)
        rho = pm.Uniform('rho', -1, 1)
        #cov = pm.math.stack([[sigma[0]**2, rho*sigma[0]*sigma[1]],
        #                     [rho*sigma[0]*sigma[1], sigma[1]**2]])
        cov = pm.math.stack([[sigma**2, rho*sigma**2],
                             [rho*sigma**2, sigma**2]])
        mu = pm.math.stack([F1**2, F2**2])
        pm.MvNormal('I_obs', mu=mu, cov=cov, observed=Y)
        tr = pm.sample(draws, tune=tune, chains=2, target_accept=0.9, progressbar=False)
    return tr

rhos = np.linspace(0.0, 0.95, 6)
w_uni_list = []
w_mv_list  = []
obs_r_list = []
dF_uni_list = []
dF_mv_list  = []

for rt in rhos:
    I1_s, I2_s = simulate_with_rho(n_pairs=25, rho_target=rt, seed=100+int(rt*100))
    obs_r = np.corrcoef(I1_s, I2_s)[0,1]
    obs_r_list.append(obs_r)

    tr_u = fit_uni(I1_s, I2_s, draws=1000, tune=500)
    f1_u = az.extract(tr_u, var_names=['F1']).to_dataframe()
    f2_u = az.extract(tr_u, var_names=['F2']).to_dataframe()
    dF_u=np.median(np.array(f2_u)-np.array(f1_u))
    tr_m = fit_mv(I1_s, I2_s, draws=1000, tune=500)
    f1_m = az.extract(tr_m, var_names=['F1']).to_dataframe()
    f2_m = az.extract(tr_m, var_names=['F2']).to_dataframe()
    dF_m=np.median(np.array(f2_m)-np.array(f1_m))

    dF_uni_list.append(dF_u)
    dF_mv_list.append(dF_m)
    w_uni_list.append(delta_hdi_width(tr_u))
    w_mv_list.append(delta_hdi_width(tr_m))

plt.figure()
plt.plot(rhos, w_uni_list, marker="o", label="univariate")
plt.plot(rhos, w_mv_list, marker="s", label="multivariate")
plt.xlabel("target correlation ρ")
plt.ylabel("95% HDI width of ΔF")
plt.title("Uncertainty in ΔF vs correlation strength")
plt.legend()
plt.grid(True)
plt.show()


plt.figure()
plt.plot(rhos, dF_uni_list, marker="^")
plt.plot(rhos, dF_mv_list, marker="^")
plt.xlabel("target correlation ρ")
plt.ylabel("Median DeltaF, true value is 0.6")
plt.title("Median DeltaF")
plt.grid(True)
plt.show()



## 6. Reporting

Include the following in a short report if you use this notebook in teaching:
- Scatter plots and correlation analysis for simulated data
- Posterior summaries and diagnostics for both models
- Comparison of 95 percent HDI for \(\Delta F\) between models
- A brief explanation of how covariance information reduces uncertainty



## 7. Notes on computation

- Sampling settings are modest for speed. Increase `draws`, `tune`, and `chains` for tighter estimates.
- Inspect trace plots and `az.summary` for convergence.
- The No U Turn Sampler in PyMC is used by default and is suitable for these models.
